# MASIVE-ALS — Docking en Kaggle (GPU gratuita)

**Procesa los PARES de ligandos** (índice par). El PC procesa los impares.

Usa el **binario Vina-GPU-2.1** (mismo motor que el PC, validado 16/08).
Bolsillos CORREGIDOS y validados (los mismos que `cribar_libreria.py` del PC).

1. Dataset `masive-als-docking-data` (ligandos + receptores)
2. Entorno → GPU
3. Ejecutar celdas en orden (o Run All)

In [ ]:
!nvidia-smi
print('GPU OK')

In [ ]:
# CELDA 2: Obtener binario Vina-GPU (descarga directa, sin pip)
# Vina-GPU-2.1 necesita libboost_program_options.so.1.74.0 (Kaggle ya no la trae)
import os, glob, subprocess, urllib.request, tarfile, shutil

# Instalar boost 1.74 (dependencia de Vina-GPU-2.1)
r = subprocess.run(['apt-get', 'update', '-qq'], capture_output=True, text=True)
r = subprocess.run(['apt-get', 'install', '-y', '-qq', 'libboost-program-options1.74.0'], capture_output=True, text=True)
print('boost instalado rc=%d' % r.returncode)
if r.returncode != 0:
    print("fallo apt, intento via pip...")
    subprocess.run(["pip", "install", "-q", "libboost"], capture_output=True, text=True)

VINA_GPU_URL = 'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/vinagpu_linux.tar.gz'
BIN_DIR = '/kaggle/working/vinagpu_linux'
os.makedirs(BIN_DIR, exist_ok=True)

def buscar_binario(base):
    if not base or not os.path.exists(base):
        return None
    hits = glob.glob(base + '/**/AutoDock-Vina-GPU-2-1', recursive=True)
    return hits[0] if hits else None

VINA_GPU_BIN = buscar_binario(BIN_DIR)
if not VINA_GPU_BIN:
    pkg = '/tmp/vinagpu_linux.tar.gz'
    urllib.request.urlretrieve(VINA_GPU_URL, pkg)
    with tarfile.open(pkg) as t:
        t.extractall(BIN_DIR)
    VINA_GPU_BIN = buscar_binario(BIN_DIR)
if not VINA_GPU_BIN:
    raise SystemExit('ERROR: no se pudo obtener Vina-GPU')
BIN_DIR = os.path.dirname(VINA_GPU_BIN)
os.chmod(VINA_GPU_BIN, 0o755)
print('Vina-GPU listo:', VINA_GPU_BIN)
print('Kernels:', len(glob.glob(BIN_DIR + '/OpenCL/src/kernels/*.cl')))

In [ ]:
# CELDA 3: Extraer datos del dataset
import os, glob, tarfile, shutil

WORK = '/kaggle/working/masive_als'
IN = '/kaggle/input/masive-als-docking-data'
for sub in ['receptores', 'ligandos', 'resultados', 'checkpoint']:
    os.makedirs(WORK + '/' + sub, exist_ok=True)

def es_receptor(nombre):
    return nombre in ('TDP43.pdbqt', 'SOD1.pdbqt', 'FUS.pdbqt')

# Kaggle descomprime los tars del dataset; copiar cualquier .pdbqt
for raiz, _, archivos in os.walk(IN):
    for a in archivos:
        if a.endswith('.pdbqt'):
            destino = WORK + '/receptores/' if es_receptor(a) else WORK + '/ligandos/'
            shutil.copy(os.path.join(raiz, a), destino + a)

print('Receptores:', len(glob.glob(WORK + '/receptores/*.pdbqt')))
print('Ligandos:', len(glob.glob(WORK + '/ligandos/*.pdbqt')))

In [ ]:
# CELDA 4: Receptores con BOLSILLOS CORREGIDOS (mismos que el PC)
RECEPTORES = {
    'TDP43': {'archivo': WORK + '/receptores/TDP43.pdbqt', 'centro': [16.3, 41.1, 48.5], 'tamano': [24, 24, 24]},
    'SOD1':  {'archivo': WORK + '/receptores/SOD1.pdbqt',  'centro': [46.5, 80.0, 73.3], 'tamano': [22, 22, 22]},
    'FUS':   {'archivo': WORK + '/receptores/FUS.pdbqt',   'centro': [-14.5, 15.1, -7.8], 'tamano': [25, 25, 25]},
}
print('Receptores:', list(RECEPTORES.keys()))
for t, info in RECEPTORES.items():
    print(' ', t, 'existe' if os.path.exists(info['archivo']) else 'FALTA')

In [ ]:
# CELDA 5: Pipeline de docking con Vina-GPU (PARES)
import csv, glob, os, time, subprocess

LIG_DIR = WORK + '/ligandos'
OUT_DIR = WORK + '/resultados'
CSV = OUT_DIR + '/resultados_kaggle.csv'
CKPT = OUT_DIR + '/hechos.txt'

if not os.path.exists(CSV):
    with open(CSV, 'w', newline='') as f:
        csv.writer(f).writerow(['ligand', 'target', 'energy', 'timestamp'])

hechos = set()
if os.path.exists(CKPT):
    hechos = set(l.strip() for l in open(CKPT) if l.strip())
print('Hechos antes:', len(hechos))

ligandos = sorted(glob.glob(LIG_DIR + '/*.pdbqt'))
print('Ligandos totales:', len(ligandos))

# TOMAR SOLO PARES (el PC procesa los impares)
ligandos = [l for i, l in enumerate(ligandos) if i % 2 == 0]
print('Asignados a Kaggle (pares):', len(ligandos))

SEEDS = [42, 2026, 777]  # 3 semillas, mejor afinidad

def acoplar(lig, target, seed, thread=8000):
    info = RECEPTORES[target]
    cfg = '/tmp/cfg_%s_%s.txt' % (os.path.basename(lig).replace('.pdbqt', '')[:20], target)
    with open(cfg, 'w') as f:
        f.write('receptor = %s\n' % info['archivo'])
        f.write('ligand = %s\n' % lig)
        f.write('center_x = %s\n' % info['centro'][0])
        f.write('center_y = %s\n' % info['centro'][1])
        f.write('center_z = %s\n' % info['centro'][2])
        f.write('size_x = %s\n' % info['tamano'][0])
        f.write('size_y = %s\n' % info['tamano'][1])
        f.write('size_z = %s\n' % info['tamano'][2])
        f.write('num_modes = 3\n')
        f.write('seed = %d\n' % seed)
        f.write('thread = %d\n' % thread)
    try:
        r = subprocess.run([VINA_GPU_BIN, '--config', cfg], capture_output=True, text=True,
                           timeout=1800, cwd=BIN_DIR)
        out = (r.stdout or '') + (r.stderr or '')
        if r.returncode != 0:
            return None, 'rc=%d %s' % (r.returncode, out[-150:])
        for ln in out.splitlines():
            s = ln.split()
            if len(s) >= 2 and s[0] == '1':
                try:
                    return round(float(s[1]), 4), None
                except ValueError:
                    pass
        return None, 'sin afinidad: ' + out[-150:]
    except Exception as ex:
        return None, str(ex)[:100]

def tiene_atomos(lig):
    try:
        with open(lig, errors='replace') as f:
            return any(l.startswith(('ATOM', 'HETATM')) for l in f)
    except Exception:
        return False

t0 = time.time()
n_ok = 0
n_err = 0
for lig in ligandos:
    nombre = os.path.basename(lig).replace('.pdbqt', '')
    if nombre in hechos:
        continue
    if not tiene_atomos(lig):
        print('LIGANDO_VACIO', nombre)
        hechos.add(nombre)
        continue
    for target in RECEPTORES:
        energias = []
        err = None
        for sd in SEEDS:
            e, er = acoplar(lig, target, sd)
            if e is not None:
                energias.append(e)
            else:
                err = er
        if energias:
            energia = min(energias)
            with open(CSV, 'a', newline='') as f:
                csv.writer(f).writerow([nombre, target, energia, time.strftime('%Y-%m-%d %H:%M:%S')])
            n_ok += 1
        else:
            n_err += 1
            print('ERROR', nombre, target, err)
    with open(CKPT, 'a') as f:
        f.write(nombre + '\n')
    hechos.add(nombre)
    if n_ok % 5 == 0 and n_ok > 0:
        print('[%d acoplados, %d errores] %.1f min' % (n_ok, n_err, (time.time() - t0) / 60), flush=True)

print()
print('=== TANDA COMPLETADA ===')
print('Acoplados:', n_ok, '| Errores:', n_err)
print('CSV:', CSV)

In [ ]:
# CELDA 6: Resumen y descarga
import csv
rows = list(csv.DictReader(open(CSV)))
print('Total resultados:', len(rows))
if rows:
    best = sorted(rows, key=lambda x: float(x['energy']))[:10]
    print()
    print('Top 10:')
    for r in best:
        print('  ', r['ligand'], r['target'], r['energy'])
print()
print('Descargar: resultados_kaggle.csv')